In [ ]:
# 1) Mount Google Drive and clone the repo if needed
from pathlib import Path
import os
import shutil
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('Drive mount skipped or already mounted:', exc)

repo_path = Path('/content/project')
if repo_path.exists():
    shutil.rmtree(repo_path, ignore_errors=True)

clone_success = False
for url in ['https://github.com/hanhwannau/A.git', 'https://github.com/hanhwannau/A']:
    for branch in ['master', 'main']:
        print(f'Trying clone from {url} branch {branch}...')
        try:
            subprocess.run(
                ['git', 'clone', '--depth', '1', '--branch', branch, url, str(repo_path)],
                check=True,
                capture_output=True,
                text=True,
            )
            print('Clone succeeded')
            clone_success = True
            break
        except subprocess.CalledProcessError as exc:
            print('Clone failed:', exc.returncode)
            print('stdout:', exc.stdout)
            print('stderr:', exc.stderr)
    if clone_success:
        break

if not clone_success:
    raise RuntimeError('Failed to clone repository from GitHub. Please check repo access and URL.')

print('Repository cloned to:', repo_path)
print('Files in app/plugins:', [p.name for p in (repo_path / 'app' / 'plugins').glob('*')])


# One-click Colab video pipeline

Chỉ cần chạy từng cell theo thứ tự bên dưới trong Google Colab. Mỗi bước đã được tách rõ ràng:

1. Mount Google Drive và clone repo
2. Cài dependencies
3. Khởi Gradio UI để upload video và chạy backend

Notebook này dùng repo `master` nếu có, fallback sang `main` nếu cần.

In [ ]:
# 2) Install ffmpeg and Python dependencies
import sys
import subprocess

subprocess.run(['apt-get', 'update', '-y'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], check=True)
subprocess.run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '-r',
    '/content/project/requirements.txt',
    'gradio',
], check=True)
print('Dependencies installed.')

In [ ]:
# 3) Run the Gradio UI for the video pipeline
import sys
import os
from pathlib import Path

repo_path = Path('/content/project')
if not repo_path.exists() or not (repo_path / 'app' / 'plugins' / 'runner.py').exists():
    raise FileNotFoundError('Repo not found or invalid. Please run the first cell again.')

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

import gradio as gr
from app.plugins.runner import run_from_config


def process(video_file):
    if video_file is None:
        return 'No video uploaded', '', 'Upload a video file first.'

    video_path = Path(video_file) if isinstance(video_file, (str, Path)) else Path(getattr(video_file, 'name', str(video_file)))
    if not video_path.exists():
        return 'failed', '', f'Video file not found: {video_path}'

    print('Running pipeline for', video_path)
    result = run_from_config('config_pipeline_full.yaml', video_url=str(video_path))
    rendered_path = Path(result.get('rendered_path', 'outputs/final.mp4'))
    if not rendered_path.is_absolute():
        rendered_path = repo_path / rendered_path

    if rendered_path.exists():
        return 'success', str(rendered_path), 'Rendered output is ready.'

    return 'failed', '', f'Output not found. {result.get("render_error", "")}'

with gr.Blocks() as demo:
    gr.Markdown('# One-click Colab Video Pipeline')
    gr.Markdown('Upload a video and click Run. The backend will execute on Colab.')
    video_input = gr.File(label='Upload video', file_count='single', type='filepath')
    status = gr.Textbox(label='Status', interactive=False)
    output_path = gr.Textbox(label='Rendered output path', interactive=False)
    message = gr.Textbox(label='Message', interactive=False)
    run_button = gr.Button('Run pipeline')
    run_button.click(process, inputs=[video_input], outputs=[status, output_path, message])

demo.launch(share=True)
